In [ ]:
%pip install azure-identity azure-mgmt-costmanagement azure.mgmt.costmanagement
dbutils.library.restartPython()

In [ ]:
from datetime import datetime, timezone, timedelta
from pyspark.sql import functions as F
from pyspark.sql import Row
import time
import json
import requests
import logging
from azure.identity import ClientSecretCredential
from azure.mgmt.costmanagement import CostManagementClient
from azure.mgmt.costmanagement.models import (
    QueryDefinition,
    QueryDataset,
    QueryTimePeriod,
    QueryAggregation,
    QueryGrouping,
    ExportType,
    TimeframeType,
)

In [ ]:
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "")
dbutils.widgets.text("overlap_days", "3")
dbutils.widgets.text("subscription_id", "")
dbutils.widgets.text("scope", "")

In [ ]:
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
overlap_days = int(dbutils.widgets.get("overlap_days") or "3")

In [ ]:
# Disable Azure SDK verbose logs
logging.getLogger("azure").setLevel(logging.WARNING)
logging.getLogger("azure.core").setLevel(logging.WARNING)
logging.getLogger("azure.identity").setLevel(logging.WARNING)
logging.getLogger("py4j").setLevel(logging.ERROR)

In [ ]:
audit_table = f"{catalog}.{schema}.dbspend360_audit_log"
target_table = f"{catalog}.{schema}.dbspend360_cloud_cost_explorer"
error_log_table = f"{catalog}.{schema}.dbspend360_error_log"
breakdown_table = f"{catalog}.{schema}.dbspend360_other_cost_breakdown"

In [ ]:
class AzureCostClient:
    def __init__(self, subscription_id, tenant_id, client_id, client_secret):
        self.subscription_id = subscription_id

        self.credential = ClientSecretCredential(
            tenant_id=tenant_id,
            client_id=client_id,
            client_secret=client_secret
        )

        self.client = CostManagementClient(self.credential)
        self.scope = f"/subscriptions/{self.subscription_id}"
        self.max_chunk_days = 10
        self.max_retries = 3

    # -------- Public API --------
    def group_by_job_clusterid_daily(
        self,
        start_date: datetime,
        end_date: datetime,
        tag_name: str = "clusterid",
    ):
        """Entry point: handles chunking and unions all results.

        Uses dual grouping (TagKey + MeterCategory) to get per-service
        cost breakdown per cluster per day in a single API call.
        """
        start_utc, end_utc = self._to_utc(start_date, end_date)

        chunks = self._build_chunks(start_utc, end_utc, self.max_chunk_days)

        all_chunk_dfs = []
        for chunk_start, chunk_end in chunks:
            print(f"Querying chunk {chunk_start} → {chunk_end}")
            df = self._query_with_retries(chunk_start, chunk_end, tag_name)
            if df is not None and df.limit(1).count() > 0:
                all_chunk_dfs.append(df)
            time.sleep(5)

        if not all_chunk_dfs:
            return None

        result_df = all_chunk_dfs[0]
        for df in all_chunk_dfs[1:]:
            result_df = result_df.unionByName(df, allowMissingColumns=True)

        return result_df

    # -------- Helpers: date & chunks --------
    def _to_utc(self, start_date: datetime, end_date: datetime):
        start_utc = start_date.astimezone(timezone.utc)
        end_utc = end_date.astimezone(timezone.utc)
        return start_utc, end_utc

    def _build_chunks(self, start_utc: datetime, end_utc: datetime, max_days: int):
        """Return list of (chunk_start, chunk_end) in UTC."""
        chunks = []
        current = start_utc
        while current <= end_utc:
            chunk_end = min(current + timedelta(days=max_days - 1), end_utc)
            chunks.append((current, chunk_end))
            current = chunk_end + timedelta(days=1)
        return chunks

    # -------- Helpers: query construction --------
    def _build_dataset(self, tag_name: str):
        return QueryDataset(
            granularity="Daily",
            aggregation={"totalCost": QueryAggregation(name="Cost", function="Sum")},
            grouping=[
                QueryGrouping(type="TagKey", name=tag_name),
                QueryGrouping(type="Dimension", name="MeterCategory"),
            ],
        )

    def _build_query_definition(self, start_utc: datetime, end_utc: datetime, dataset):
        return QueryDefinition(
            type=ExportType.ACTUAL_COST,
            timeframe=TimeframeType.CUSTOM,
            time_period=QueryTimePeriod(from_property=start_utc, to=end_utc),
            dataset=dataset,
        )

    def _build_query_body_json(self, start_utc: datetime, end_utc: datetime, tag_name: str):
        body = {
            "type": "ActualCost",
            "timeframe": "Custom",
            "timePeriod": {
                "from": start_utc.isoformat(),
                "to": end_utc.isoformat(),
            },
            "dataset": {
                "granularity": "Daily",
                "aggregation": {
                    "totalCost": {
                        "name": "Cost",
                        "function": "Sum",
                    }
                },
                "grouping": [
                    {"type": "TagKey", "name": tag_name},
                    {"type": "Dimension", "name": "MeterCategory"},
                ],
            },
        }
        return json.dumps(body)

    # -------- Core call with retries --------
    def _query_with_retries(self, start_utc, end_utc, tag_name):
        dataset = self._build_dataset(tag_name)
        query = self._build_query_definition(start_utc, end_utc, dataset)
        query_json = self._build_query_body_json(start_utc, end_utc, tag_name)

        attempt = 0
        last_exception = None

        while attempt < self.max_retries:
            try:
                return self._execute_query(query, query_json)
            except Exception as e:
                last_exception = e
                attempt += 1

                is_429 = "429" in str(e) or "Too many requests" in str(e)
                if attempt >= self.max_retries:
                    break

                if is_429:
                    wait_sec = 30 * attempt
                    print(f"Rate limited (429) on main query, waiting {wait_sec}s (attempt {attempt})...")
                    time.sleep(wait_sec)
                else:
                    wait_sec = 2 ** attempt
                    print(f"Error on main query, waiting {wait_sec}s (attempt {attempt})...")
                    time.sleep(wait_sec)

        raise last_exception

    # -------- Single query + pagination --------
    def _execute_query(self, query, query_json: str):
        result = self.client.query.usage(self.scope, parameters=query)

        if not result.rows:
            return None

        # Read column names from the API response for robustness
        col_names = None
        if result.columns:
            col_names = [col.name for col in result.columns]

        all_rows = list(result.rows)

        next_link = getattr(result, "next_link", None)
        token = None
        if next_link:
            token = self.credential.get_token(
                "https://management.azure.com/.default"
            ).token

        while next_link:
            next_link, page_rows = self._fetch_next_page(next_link, token, query_json)
            all_rows.extend(page_rows)
            if next_link:
                time.sleep(2)

        return self._rows_to_df(all_rows, col_names)

    def _fetch_next_page(self, next_link: str, token: str, query_json: str):
        while True:
            resp = requests.post(
                next_link,
                headers={
                    "Authorization": f"Bearer {token}",
                    "Content-Type": "application/json",
                },
                data=query_json,
            )

            if resp.status_code == 429:
                headers = resp.headers
                retry_after = (
                    headers.get("x-ms-ratelimit-microsoft.costmanagement-qpu-retry-after")
                    or headers.get("x-ms-ratelimit-microsoft.costmanagement-entity-retry-after")
                    or headers.get("x-ms-ratelimit-microsoft.costmanagement-tenant-retry-after")
                    or headers.get("x-ms-ratelimit-microsoft.costmanagement-client-retry-after")
                    or headers.get("Retry-After")
                )

                wait_sec = int(retry_after) if retry_after is not None else 30
                print(f"429 throttled, waiting {wait_sec}s before retrying nextLink...")
                time.sleep(wait_sec)
                continue

            resp.raise_for_status()
            data = resp.json()
            props = data.get("properties", {})
            page_rows = props.get("rows", [])
            new_next_link = props.get("nextLink")
            return new_next_link, page_rows

    # -------- Helper: convert rows to DataFrame --------
    def _rows_to_df(self, rows, col_names=None):
        """Build a Spark DataFrame from Azure response rows.

        With dual grouping (TagKey + MeterCategory), the response typically has
        6 columns: [Cost, UsageDate, TagKey, ClusterIdValue, MeterCategory, Currency].
        We use API-reported column names when available, falling back to hardcoded
        names matching the original 5-column format for backward safety.
        """
        if col_names and len(col_names) == len(rows[0]) if rows else False:
            df = spark.createDataFrame(rows, col_names)
            # Normalize column names to lowercase for consistent downstream access
            for c in df.columns:
                df = df.withColumnRenamed(c, c.lower())
            return df

        # Fallback: 6 columns for dual grouping, 5 for legacy single grouping
        if rows and len(rows[0]) == 6:
            return spark.createDataFrame(
                rows,
                ["cost", "date_key", "tag_key", "cluster_id", "meter_category", "currency"],
            )

        return spark.createDataFrame(
            rows,
            ["cost", "date_key", "tag_key", "cluster_id", "currency"],
        )


In [ ]:
# =======================================================
# Azure MeterCategory Classification Framework
# =======================================================
# Single source of truth for Azure MeterCategory classification.
# Category -> list of case-insensitive substring patterns.
# Any MeterCategory not matching a pattern is routed to "other"
# (never silently assigned to compute).
AZURE_METER_CLASSIFICATION = {
    "compute": ["virtual machine"],
    "storage": ["storage", "disk"],
    "network": ["bandwidth", "virtual network", "load balancer", "network watcher"],
}

logging.basicConfig(level=logging.INFO)
azure_logger = logging.getLogger("AzureCostExplorer")


def classify_azure_meter_category(meter_category: str) -> str:
    """Classify an Azure MeterCategory string using AZURE_METER_CLASSIFICATION.

    Unknown categories return 'other' -- never silently 'compute'.
    """
    if not meter_category:
        return "other"
    lower = meter_category.lower()
    for category, patterns in AZURE_METER_CLASSIFICATION.items():
        for pattern in patterns:
            if pattern in lower:
                return category
    return "other"


def build_azure_category_column(mc_col_name: str):
    """Build a PySpark Column expression from AZURE_METER_CLASSIFICATION.

    Generates the when/otherwise chain programmatically so the dict
    is the sole source of truth. Unknown meters map to 'other'.
    """
    lower_mc = F.lower(F.col(mc_col_name))
    expr = None
    for category, patterns in AZURE_METER_CLASSIFICATION.items():
        cond = lower_mc.contains(patterns[0])
        for p in patterns[1:]:
            cond = cond | lower_mc.contains(p)
        if expr is None:
            expr = F.when(cond, F.lit(category))
        else:
            expr = expr.when(cond, F.lit(category))
    return expr.otherwise(F.lit("other"))


# =======================================================
# APP
# =======================================================
class AzureCostReporterApp:
    """Orchestrates incremental Azure cost ingestion into the cloud cost table.

    Invariant enforced: cloud_cost = compute_cost + storage_cost + network_cost + other_cost
    """

    def __init__(self):
        scope=dbutils.widgets.get("scope")
        subscription_id=dbutils.widgets.get("subscription_id")
        tenant_id=dbutils.secrets.get(scope, "tenant_id")
        client_id=dbutils.secrets.get(scope, "client_id")
        client_secret= dbutils.secrets.get(scope, "client_secret")
        
        self.client = AzureCostClient(
            subscription_id,
            tenant_id,
            client_id,
            client_secret,
        )

    def run(self):

        wm = (
            spark.table(audit_table)
                 .filter("table_name = 'dbspend360_cloud_cost_explorer' AND status = 'SUCCESS'")
        )

        if wm.limit(1).count() == 0:
            last_end_date = datetime.now(timezone.utc).date() - timedelta(days=365-overlap_days)
        else:
            last_end_date = wm.agg(F.max("end_date")).collect()[0][0]

        start_dt = last_end_date - timedelta(days=overlap_days - 1)
        end_dt = datetime.now(timezone.utc).date()

        print(f"Querying Azure cost from {start_dt} to {end_dt} (overlap_days={overlap_days})")

        if start_dt > end_dt:
            message = (
                f"Invalid date window: start_dt={start_dt} > end_dt={end_dt}. "
                f"Check audit table and overlap_days={overlap_days}."
            )

            run_log_df = spark.createDataFrame([
                Row(
                    table_name="dbspend360_cloud_cost_explorer",
                    start_date=start_dt,
                    end_date=end_dt,
                    status="FAILED",
                    row_count=0,
                    message=message,
                    created_at=datetime.now(timezone.utc),
                )
            ])
            run_log_df.write.mode("append").insertInto(audit_table)
            dbutils.notebook.exit("FAILED: Invalid date window.")

        self._ensure_schema_columns()

        spark_df = self.client.group_by_job_clusterid_daily(
            start_date=datetime.combine(start_dt, datetime.min.time(), tzinfo=timezone.utc),
            end_date=datetime.combine(end_dt, datetime.max.time(), tzinfo=timezone.utc),
            tag_name="clusterid",
        )

        quality_msg = f"overlap_days={overlap_days}"

        if spark_df is None or spark_df.limit(1).count() == 0:
            print("No Azure cost data returned by API for the requested range.")
            merged_row_count = 0
        else:
            # Resolve date_key column (API may return "usagedate" or "date_key")
            date_col = "usagedate" if "usagedate" in [c.lower() for c in spark_df.columns] else "date_key"
            spark_df = spark_df.withColumn(
                "cost_incurred_date",
                F.to_date(F.col(date_col).cast("string"), "yyyyMMdd"),
            )

            # Resolve cluster_id column (API may return "clusterid" as tag value column)
            if "cluster_id" not in spark_df.columns:
                tag_value_col = next(
                    (c for c in spark_df.columns if c.lower() not in
                     ("cost", date_col.lower(), "tag_key", "meter_category",
                      "metercategory", "currency", "cost_incurred_date")),
                    None,
                )
                if tag_value_col:
                    spark_df = spark_df.withColumnRenamed(tag_value_col, "cluster_id")

            inc_df = (
                spark_df
                .filter((F.col("cluster_id").isNotNull()) & (F.col("cluster_id") != ""))
                .filter(F.col("cost_incurred_date").isNotNull())
            )

            if inc_df.limit(1).count() == 0:
                print("No incremental rows after filtering by cluster_id and cost_incurred_date.")
                merged_row_count = 0
            else:
                has_meter = "meter_category" in [c.lower() for c in inc_df.columns]
                if has_meter:
                    self._log_unclassified_meters(inc_df)
                    self._write_other_cost_breakdown(inc_df)
                    agg_df = self._classify_and_aggregate(inc_df)
                else:
                    azure_logger.warning(
                        "MeterCategory column not found; all cost assigned to cloud_cost only."
                    )
                    agg_df = (
                        inc_df
                        .groupBy("cluster_id", "currency", "cost_incurred_date")
                        .agg(F.sum("cost").alias("cloud_cost"))
                        .withColumn("compute_cost", F.lit(None).cast("double"))
                        .withColumn("storage_cost", F.lit(None).cast("double"))
                        .withColumn("network_cost", F.lit(None).cast("double"))
                        .withColumn("other_cost", F.lit(None).cast("double"))
                        .withColumn("created_at", F.current_timestamp())
                        .withColumn("updated_at", F.current_timestamp())
                    )

                merged_row_count = agg_df.count()

                quality_msg = self._compute_quality_metrics(agg_df, merged_row_count)

                agg_df.createOrReplaceTempView("cloud_cost_inc")

                spark.sql(f"""
                MERGE INTO {target_table} AS t
                USING cloud_cost_inc AS s
                ON  t.cluster_id = s.cluster_id
                AND t.currency = s.currency
                AND t.cost_incurred_date = s.cost_incurred_date
                WHEN MATCHED THEN
                  UPDATE SET
                    t.cloud_cost      = s.cloud_cost,
                    t.compute_cost    = s.compute_cost,
                    t.storage_cost    = s.storage_cost,
                    t.network_cost    = s.network_cost,
                    t.other_cost      = s.other_cost,
                    t.updated_at      = current_timestamp()
                WHEN NOT MATCHED THEN
                  INSERT (cluster_id, cloud_cost, compute_cost, storage_cost, network_cost, other_cost,
                          currency, cost_incurred_date, created_at, updated_at)
                  VALUES (s.cluster_id, s.cloud_cost, s.compute_cost, s.storage_cost, s.network_cost, s.other_cost,
                          s.currency, s.cost_incurred_date,
                          current_timestamp(), current_timestamp())
                """)

        print(f"Merged {merged_row_count} rows into {target_table} for {start_dt} → {end_dt} (overlap_days={overlap_days}).")

        run_log_df = spark.createDataFrame([
            Row(
                table_name="dbspend360_cloud_cost_explorer",
                start_date=start_dt,
                end_date=end_dt,
                status="SUCCESS",
                row_count=int(merged_row_count),
                message=quality_msg,
                created_at=datetime.now(timezone.utc),
            )
        ])

        run_log_df.write.mode("append").insertInto(audit_table)

    def _ensure_schema_columns(self):
        """Add cost segmentation columns if they don't exist yet."""
        existing = {c.name for c in spark.table(target_table).schema}
        missing = [c for c in ("compute_cost", "storage_cost", "network_cost", "other_cost") if c not in existing]
        if missing:
            cols_sql = ", ".join(f"{c} DOUBLE" for c in missing)
            spark.sql(f"ALTER TABLE {target_table} ADD COLUMNS ({cols_sql})")
            azure_logger.info(f"Added columns to {target_table}: {missing}")

    @staticmethod
    def _log_unclassified_meters(df):
        """Log unclassified MeterCategory values to both logger and error_log table."""
        mc_col = next(c for c in df.columns if c.lower() == "meter_category")
        meter_costs = (
            df.groupBy(mc_col)
            .agg(
                F.sum("cost").alias("total_cost"),
                F.count("*").alias("row_count"),
            )
            .collect()
        )
        unknown_rows = [
            r for r in meter_costs
            if r[0] and classify_azure_meter_category(r[0]) == "other"
        ]

        if not unknown_rows:
            return

        meter_names = [r[0] for r in unknown_rows]
        azure_logger.warning(f"Unclassified Azure MeterCategory values (routed to other_cost): {meter_names}")

        try:
            error_records = [
                Row(
                    source_system="AZURE",
                    error_type="UNCLASSIFIED_COST",
                    cluster_id=None,
                    job_id=None,
                    run_id=None,
                    usage_date=None,
                    currency=None,
                    error_detail=f"Unclassified meter: {r[0]}, total_cost=${r.total_cost:.4f}, rows={r.row_count}",
                    raw_record=None,
                    created_at=datetime.now(timezone.utc),
                )
                for r in unknown_rows
            ]
            spark.createDataFrame(error_records).write.mode("append").insertInto(error_log_table)
        except Exception as e:
            azure_logger.warning(f"Failed to write unclassified meters to error_log: {e}")

    def _write_other_cost_breakdown(self, inc_df):
        """Write per-meter detail for 'other' category costs to the breakdown table.

        Aggregates unclassified MeterCategory costs by (date, cluster, meter)
        and MERGEs into dbspend360_other_cost_breakdown for drilldown queries.
        Idempotent: reruns update existing rows via MERGE.
        """
        mc_col = next(c for c in inc_df.columns if c.lower() == "meter_category")
        classified = inc_df.withColumn("category", build_azure_category_column(mc_col))

        other_df = (
            classified
            .filter(F.col("category") == "other")
            .groupBy("cluster_id", mc_col, "currency", "cost_incurred_date")
            .agg(F.sum("cost").alias("cost"))
            .withColumnRenamed(mc_col, "service_name")
            .withColumn("source_system", F.lit("AZURE"))
            .withColumn("created_at", F.current_timestamp())
            .withColumn("updated_at", F.current_timestamp())
        )

        if other_df.limit(1).count() == 0:
            azure_logger.info("No 'other' category costs to write to breakdown table.")
            return

        other_df.createOrReplaceTempView("other_cost_breakdown_inc")

        spark.sql(f"""
        MERGE INTO {breakdown_table} AS t
        USING other_cost_breakdown_inc AS s
        ON  t.cost_incurred_date = s.cost_incurred_date
        AND t.cluster_id = s.cluster_id
        AND t.source_system = s.source_system
        AND t.service_name = s.service_name
        AND t.currency = s.currency
        WHEN MATCHED THEN
          UPDATE SET t.cost = s.cost, t.updated_at = current_timestamp()
        WHEN NOT MATCHED THEN
          INSERT (cost_incurred_date, cluster_id, source_system, service_name,
                  cost, currency, created_at, updated_at)
          VALUES (s.cost_incurred_date, s.cluster_id, s.source_system, s.service_name,
                  s.cost, s.currency, current_timestamp(), current_timestamp())
        """)

        breakdown_count = other_df.count()
        azure_logger.info(f"Wrote {breakdown_count} other cost breakdown rows to {breakdown_table}")

    @staticmethod
    def _classify_and_aggregate(inc_df):
        """Classify MeterCategory rows using the centralized AZURE_METER_CLASSIFICATION.

        Uses build_azure_category_column() so the dict is the single source of truth.
        Unknown meters are classified as 'other' -- never silently as 'compute'.

        Invariant: cloud_cost = compute_cost + storage_cost + network_cost + other_cost
        """
        mc_col = next(c for c in inc_df.columns if c.lower() == "meter_category")
        classified = inc_df.withColumn("category", build_azure_category_column(mc_col))

        return (
            classified
            .groupBy("cluster_id", "currency", "cost_incurred_date")
            .agg(
                F.sum(F.when(F.col("category") == "compute", F.col("cost")).otherwise(0)).alias("compute_cost"),
                F.sum(F.when(F.col("category") == "storage", F.col("cost")).otherwise(0)).alias("storage_cost"),
                F.sum(F.when(F.col("category") == "network", F.col("cost")).otherwise(0)).alias("network_cost"),
                F.sum(F.when(F.col("category") == "other", F.col("cost")).otherwise(0)).alias("other_cost"),
            )
            .withColumn("cloud_cost", F.col("compute_cost") + F.col("storage_cost") + F.col("network_cost") + F.col("other_cost"))
            .withColumn("created_at", F.current_timestamp())
            .withColumn("updated_at", F.current_timestamp())
        )

    @staticmethod
    def _compute_quality_metrics(agg_df, row_count):
        """Compute and log data quality / classification coverage metrics."""
        metrics = agg_df.agg(
            F.sum("cloud_cost").alias("total_cost"),
            F.sum("compute_cost").alias("classified_compute"),
            F.sum("storage_cost").alias("classified_storage"),
            F.sum("network_cost").alias("classified_network"),
            F.sum("other_cost").alias("unclassified_cost"),
        ).collect()[0]

        total = float(metrics.total_cost or 0)
        unclassified = float(metrics.unclassified_cost or 0)
        classified = total - unclassified
        coverage_pct = (classified / total * 100) if total > 0 else 100.0

        msg = (
            f"overlap_days={overlap_days}, rows={row_count}, "
            f"classification_coverage={coverage_pct:.1f}%, "
            f"classified_cost={classified:.2f}, "
            f"unclassified_cost={unclassified:.2f}, "
            f"total_cost={total:.2f}"
        )
        azure_logger.info(f"Data quality: {msg}")
        return msg

In [ ]:
# =======================================================
# Execute
# =======================================================
app = AzureCostReporterApp()
app.run()